# 04 - Wildfire Risk Scoring

This notebook calculates a 0-100 wildfire risk score for each 1-degree latitude-longitude grid cell.
The score combines hotspot density, average FRP, brightness, confidence, and recency. Missing
metrics are skipped automatically.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_cleaned_data
from risk_model import DEFAULT_WEIGHTS, build_and_save_risk_scores, compute_risk_scores

## Score formula

In [2]:
DEFAULT_WEIGHTS

{'hotspot_density_score': 0.35,
 'frp_score': 0.2,
 'brightness_score': 0.15,
 'confidence_score_component': 0.15,
 'recent_activity_score': 0.15}

Risk categories:

- 0-25: Low
- 26-50: Medium
- 51-75: High
- 76-100: Critical

In [3]:
df = load_cleaned_data(ROOT / "data" / "processed" / "cleaned_wildfire_data.csv")
risk_scores, risk_path = build_and_save_risk_scores(
    df,
    ROOT / "data" / "processed" / "wildfire_risk_scores.csv",
    grid_size=1.0,
)
risk_path

WindowsPath('d:/Codingan Pribadi/SERIUS/NASA FIRM/wildfire-risk-intelligence/data/processed/wildfire_risk_scores.csv')

## Top risk areas

In [4]:
columns = [
    "grid_label",
    "risk_score",
    "risk_category",
    "hotspot_count",
    "avg_frp",
    "avg_brightness",
    "avg_confidence",
    "latest_detection",
]
display(risk_scores[columns].head(10))

,grid_label,risk_score,risk_category,hotspot_count,avg_frp,avg_brightness,avg_confidence,latest_detection
0,"10.0 to 11.0 lat, -14.0 to -13.0 lon",80.84,Critical,3208,44.676181,344.799585,64.008728,2026-04-25 16:37:00+00:00
1,"11.0 to 12.0 lat, -14.0 to -13.0 lon",80.38,Critical,3782,36.610518,343.299297,63.001851,2026-04-25 16:37:00+00:00
2,"8.0 to 9.0 lat, -13.0 to -12.0 lon",79.05,Critical,5183,19.796091,340.507849,63.470191,2026-04-25 16:35:00+00:00
3,"9.0 to 10.0 lat, -13.0 to -12.0 lon",78.84,Critical,2948,33.941825,342.197737,63.262890,2026-04-25 16:37:00+00:00
4,"22.0 to 23.0 lat, 93.0 to 94.0 lon",78.53,Critical,3046,35.032091,340.321779,63.280696,2026-04-25 07:35:00+00:00
5,"10.0 to 11.0 lat, -13.0 to -12.0 lon",77.77,Critical,3266,21.853337,342.965254,62.873852,2026-04-25 14:50:00+00:00
6,"43.0 to 44.0 lat, 127.0 to 128.0 lon",75.15,High,5054,7.344964,337.380857,66.640087,2026-04-25 04:53:00+00:00
7,"-15.0 to -14.0 lat, 125.0 to 126.0 lon",74.95,High,3914,10.021645,335.751778,66.095810,2026-04-25 19:04:00+00:00
8,"19.0 to 20.0 lat, -90.0 to -89.0 lon",74.91,High,1887,20.564960,338.993884,65.871754,2026-04-25 19:57:00+00:00
9,"10.0 to 11.0 lat, -12.0 to -11.0 lon",74.57,High,2311,14.563051,341.997360,62.428819,2026-04-25 14:50:00+00:00


In [5]:
category_counts = (
    risk_scores["risk_category"]
    .value_counts()
    .reindex(["Low", "Medium", "High", "Critical"])
    .dropna()
    .reset_index()
)
category_counts.columns = ["risk_category", "area_count"]
fig = px.bar(
    category_counts,
    x="risk_category",
    y="area_count",
    title="Risk category distribution",
    labels={"risk_category": "Risk category", "area_count": "Grid cells"},
)
fig.show()
display(category_counts)

,risk_category,area_count
0,Low,36
1,Medium,4508
2,High,2269
3,Critical,6


## Risk map

In [6]:
top_for_map = risk_scores.sort_values("risk_score", ascending=False).head(600)
fig = px.scatter_geo(
    top_for_map,
    lat="center_lat",
    lon="center_lon",
    size="hotspot_count",
    color="risk_category",
    hover_name="grid_label",
    hover_data=["risk_score", "hotspot_count", "avg_frp", "avg_brightness", "latest_detection"],
    projection="natural earth",
    title="Top risk grid cells",
    color_discrete_map={
        "Low": "#2e7d32",
        "Medium": "#f9a825",
        "High": "#ef6c00",
        "Critical": "#c62828",
    },
)
fig.update_layout(geo=dict(showland=True, landcolor="#f8fafc", showcountries=True))
fig.show()

## Model interpretation

The score is not a prediction of future ignition. It is a monitoring priority index based on observed
FIRMS detections. High and Critical grids indicate areas where repeated detections, stronger thermal
signals, and recent activity overlap.